# import libraries

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

In [3]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(34):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-4b-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

In [4]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-4b-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/circuit_tracer/transcoder/single_layer_transcoder.py:513: UserWarning: Lazy loading is not supported for GemmaScope2 format due to different key naming conventions. Setting lazy_encoder=False and lazy_decoder=False.
  warnings.warn(


In [5]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-pt")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-4b-pt",
    transcoders=transcoder_set,
    backend="transformerlens",
    dtype=torch.bfloat16,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-3-4b-pt into HookedTransformer


# 1. get top influential features

In [6]:
import json
import glob
from collections import defaultdict

In [7]:
GRAPH_DIR = "./graphs/gemma-3-4b"  # ← set this
DESCRIBED_LAYERS = {9, 17, 22, 29}

In [8]:
def parse_local_feat(node):
    """Extract (layer_int, local_feat_idx) from jsNodeId."""
    js = node.get("jsNodeId", "")
    try:
        layer_feat, _ = js.rsplit("-", 1)
        layer_str, feat_str = layer_feat.split("_", 1)
        return int(layer_str), int(feat_str)
    except Exception:
        return None, None

In [25]:
all_features = defaultdict(float)
 
for fpath in sorted(glob.glob(f"{GRAPH_DIR}/step-*.json")):
    with open(fpath) as f:
        data = json.load(f)
    for node in data.get("nodes", []):
        if "transcoder" not in node.get("feature_type", ""):
            continue
        layer, local_feat = parse_local_feat(node)
        if layer is None:
            continue
        inf = abs(node.get("influence", 0) or 0)
        all_features[(layer, local_feat)] += inf

### top ten from all layers

In [10]:
top_10_overall = sorted(all_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 OVERALL (all layers)")
print("=" * 70)
for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}  {in_desc}")

TOP 10 OVERALL (all layers)
 1. L29 F 1784  influence=  7.1921  ✓ DESCRIBED
 2. L32 F  156  influence=  7.1858    undescribed
 3. L29 F 2081  influence=  7.1153  ✓ DESCRIBED
 4. L29 F  107  influence=  7.0198  ✓ DESCRIBED
 5. L31 F 1432  influence=  6.9249    undescribed
 6. L28 F 3512  influence=  6.8427    undescribed
 7. L26 F 3121  influence=  6.7691    undescribed
 8. L31 F 5799  influence=  6.7016    undescribed
 9. L33 F  294  influence=  6.6289    undescribed
10. L33 F10133  influence=  6.5203    undescribed


### top ten from only described layers

In [11]:
described_features = {lf: s for lf, s in all_features.items() if lf[0] in DESCRIBED_LAYERS}
top_10_described_sorted = sorted(described_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM DESCRIBED LAYERS ONLY (9, 17, 22, 29)")
print("=" * 70)
if top_10_described_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_described_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_described = [lf for lf, _ in top_10_described_sorted]

TOP 10 FROM DESCRIBED LAYERS ONLY (9, 17, 22, 29)
 1. L29 F 1784  influence=  7.1921
 2. L29 F 2081  influence=  7.1153
 3. L29 F  107  influence=  7.0198
 4. L29 F   80  influence=  5.9334
 5. L29 F 3347  influence=  5.9326
 6. L29 F 9459  influence=  5.7328
 7. L29 F11982  influence=  5.6974
 8. L29 F 6449  influence=  5.6277
 9. L29 F   33  influence=  5.5697
10. L29 F  440  influence=  5.3364


### top ten from every other layer

In [12]:
other_features = {lf: s for lf, s in all_features.items() if lf[0] not in DESCRIBED_LAYERS}
top_10_undescribed_sorted = sorted(other_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM OTHER LAYERS (not 9, 17, 22, 29)")
print("=" * 70)
if top_10_undescribed_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_undescribed_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_undescribed = [lf for lf, _ in top_10_undescribed_sorted]

TOP 10 FROM OTHER LAYERS (not 9, 17, 22, 29)
 1. L32 F  156  influence=  7.1858
 2. L31 F 1432  influence=  6.9249
 3. L28 F 3512  influence=  6.8427
 4. L26 F 3121  influence=  6.7691
 5. L31 F 5799  influence=  6.7016
 6. L33 F  294  influence=  6.6289
 7. L33 F10133  influence=  6.5203
 8. L33 F 1372  influence=  6.4285
 9. L33 F  316  influence=  6.3146
10. L31 F 1174  influence=  6.3012


### summary

In [13]:
print(f"Total unique transcoder features: {len(all_features)}")
print(f"Features in described layers:     {len(described_features)}")
print(f"Features in other layers:         {len(other_features)}")

Total unique transcoder features: 2401
Features in described layers:     323
Features in other layers:         2078


# intervention

### all 10 features suppressed

In [14]:
# Your prompt
prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

# Target position (typically -1 for last token, or specify the rhyme token position)
TARGET_POS = -1

In [15]:
features_to_intervene = [lf for lf, _ in top_10_overall]
intervention_tuples = [(layer, TARGET_POS, feat, 0.0) for layer, feat in features_to_intervene]

print("\n" + "=" * 70)
print("INTERVENTION TUPLES (top 10 features, suppressed at target position)")
print("=" * 70)
for i, (layer, pos, feat, val) in enumerate(intervention_tuples, 1):
    print(f"{i:2d}. Layer {layer:2d}, Position {pos:3d}, Feature {feat:5d}, Value {val}")


INTERVENTION TUPLES (top 10 features, suppressed at target position)
 1. Layer 29, Position  -1, Feature  1784, Value 0.0
 2. Layer 32, Position  -1, Feature   156, Value 0.0
 3. Layer 29, Position  -1, Feature  2081, Value 0.0
 4. Layer 29, Position  -1, Feature   107, Value 0.0
 5. Layer 31, Position  -1, Feature  1432, Value 0.0
 6. Layer 28, Position  -1, Feature  3512, Value 0.0
 7. Layer 26, Position  -1, Feature  3121, Value 0.0
 8. Layer 31, Position  -1, Feature  5799, Value 0.0
 9. Layer 33, Position  -1, Feature   294, Value 0.0
10. Layer 33, Position  -1, Feature 10133, Value 0.0


In [16]:
print("\n" + "=" * 70)
print("GENERATING WITH ORIGINAL FEATURES (no intervention)")
print("=" * 70)

pre_intervention_text = model.feature_intervention_generate(
    prompt,
    [],  # empty list = no interventions
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{pre_intervention_text}")

# ─────────────────────────────────────────────────────────────
# Generate with all 20 features suppressed
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("GENERATING WITH ALL 10 FEATURES SUPPRESSED")
print("=" * 70)

post_intervention_text = model.feature_intervention_generate(
    prompt,
    intervention_tuples,
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{post_intervention_text}")


GENERATING WITH ORIGINAL FEATURES (no intervention)


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
He ate it and then he had to crap it

GENERATING WITH ALL 10 FEATURES SUPPRESSED


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
питы, питы, питы, питы


In [17]:
print("\n" + "=" * 70)
print("COMPARISON: PRE vs POST INTERVENTION")
print("=" * 70)

from circuit_tracer.utils.demo_utils import display_generations_comparison

display_generations_comparison(
    prompt,
    [pre_intervention_text],
    [post_intervention_text]
)


COMPARISON: PRE vs POST INTERVENTION


### get pseudo clerp description

In [18]:
def pseudo_clerp_topk(model, layer, local_feat, tokenizer, top_k=10):
    """
    Project feature decoder direction through unembedding matrix.
    Returns top-k predicted tokens for this feature.
    """
    tc = model.transcoders[layer]
    # W_dec is the direction in residual space the feature 'writes' to
    W_dec = tc.W_dec[local_feat] # shape: [d_model]
    
    with torch.no_grad():
        # model.unembed.W_U shape is usually [d_model, vocab_size]
        W_U = model.unembed.W_U 
        
        # We want to project W_dec (d_model) onto each vocab entry (d_model)
        # So we do: [vocab_size, d_model] @ [d_model]
        # We use W_U.T to get [vocab_size, d_model]
        logits = torch.matmul(W_U.T, W_dec.to(W_U.dtype)) # result: [vocab_size]
    
    # Get top-k tokens
    top_ids = logits.topk(top_k).indices.tolist()
    top_tokens = [tokenizer.decode([i]) for i in top_ids]
    
    return top_tokens

# ─────────────────────────────────────────────────────────────
# Run pseudo-clerp for all top 10 overall features
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print("PSEUDO-CLERP FOR TOP 10 OVERALL FEATURES")
print("=" * 70)

for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    tokens = pseudo_clerp_topk(model, layer, feat, tokenizer, top_k=10)
    
    # Check if this is a described feature
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    
    print(f"\n{i:2d}. L{layer:2d} F{feat:5d} (influence={score:8.4f}) {in_desc}")
    print(f"    Top tokens: {tokens}")

PSEUDO-CLERP FOR TOP 10 OVERALL FEATURES

 1. L29 F 1784 (influence=  7.1921) ✓ DESCRIBED
    Top tokens: [' Efter', ' För', ' Geb', ' Rie', ' Vä', ' Kas', 'Efter', ' Arbe', 'Kam', ' Stol']

 2. L32 F  156 (influence=  7.1858)   undescribed
    Top tokens: ['<unused338>', 'ꗜ', 'ri', '𒅤', ' PLDNN', '𒉶', 'rik', ' rid', '𒂷', 'MalayMarks']

 3. L29 F 2081 (influence=  7.1153) ✓ DESCRIBED
    Top tokens: ['Gre', '<strong>', '<sub>', ' Gre', ' gre', 'Gr', ' calon', ' Ref', ' ল', '<i>']

 4. L29 F  107 (influence=  7.0198) ✓ DESCRIBED
    Top tokens: ['<unused366>', '<unused687>', '<unused192>', '<unused201>', '<unused1166>', '<unused108>', '<unused1207>', 'Referências', '.</', '<unused712>']

 5. L31 F 1432 (influence=  6.9249)   undescribed
    Top tokens: [' פ', ' การ', ' ק', ' ס', ' ו', ' מ', ' ג', ' ש', ' כ', ' അ']

 6. L28 F 3512 (influence=  6.8427)   undescribed
    Top tokens: [' str', ' Lo', ' oznac', '條', 'が登場', ' Str', ' Card', 'MathML', ' Ministries', ' aggravate']

 7. L26 F 312

In [19]:
# Pseudo-clerp descriptions for top 10 overall features from Gemini 3 (fast)
pseudo_clerps = {
(29, 1784): "Swedish and Northern European capitalized sentence-starting words.",
(32, 156): "Cuneiform characters and ancient Sumerian/Akkadian script tokens.",
(29, 2081): "HTML formatting tags and structural markup elements.",
(29, 107): "Unused internal tokens and bibliography reference markers.",
(31, 1432): "Hebrew alphabet characters and Thai script prefixes.",
(28, 3512): "Mixed technical strings including string variables and MathML.",
(26, 3121): "Mathematical slash symbols and fraction division delimiters.",
(31, 5799): "Concrete singular nouns representing physical objects.",
(33, 294): "Ancient Mesopotamian pictographic and cuneiform logograms.",
(33, 10133): "Common multilingual word fragments and grammatical prefixes.",
}

### one by one suppression

In [20]:
print("Suppressing one feature at a time to isolate effects...")

ablation_results = []

for layer, feat in features_to_intervene:  # all 10 features
    single_intervention = [(layer, TARGET_POS, feat, 0.0)]
    
    try:
        ablated_text = model.feature_intervention_generate(
            prompt,
            single_intervention,
            do_sample=False,
            verbose=False
        )[0]
        
        # Check if output changed
        changed = ablated_text != pre_intervention_text
        status = "CHANGED" if changed else "unchanged"
        
        # Get clerp (try pseudo_clerps first, fall back to clerps)
        clerp_desc = pseudo_clerps.get((layer, feat)) or clerps.get((layer, feat))
        clerp_str = f" | {clerp_desc}" if clerp_desc else ""
        
        ablation_results.append({
            'layer': layer,
            'feat': feat,
            'changed': changed,
            'text': ablated_text,
            'clerp': clerp_desc
        })
        
        print(f"\nL{layer:2d} F{feat:5d}: {status}{clerp_str}")
        if changed:
            print(f"  → {ablated_text}")
    
    except Exception as e:
        print(f"L{layer:2d} F{feat:5d}: ERROR ({str(e)[:50]})")

Suppressing one feature at a time to isolate effects...

L29 F 1784: CHANGED | Swedish and Northern European capitalized sentence-starting words.
  → A rhyming couplet:
He saw a carrot and had to grab it,
如果你想吃它,你必须把它拔出来

L32 F  156: CHANGED | Cuneiform characters and ancient Sumerian/Akkadian script tokens.
  → A rhyming couplet:
He saw a carrot and had to grab it,

He ate it and then he had to crap

L29 F 2081: CHANGED | HTML formatting tags and structural markup elements.
  → A rhyming couplet:
He saw a carrot and had to grab it,
sang the song of the rabbit.
He saw

L29 F  107: CHANGED | Unused internal tokens and bibliography reference markers.
  → A rhyming couplet:
He saw a carrot and had to grab it,
tle, tle, tle, tle, tle,

L31 F 1432: CHANGED | Hebrew alphabet characters and Thai script prefixes.
  → A rhyming couplet:
He saw a carrot and had to grab it,
 Dichotomy:
The carrot was a carrot,

L28 F 3512: CHANGED | Mixed technical strings including string variables and MathML.
 

# using clerp descriptions for SAE

### get top 10 from described layers

In [21]:
described_features = {lf: s for lf, s in all_features.items() if lf[0] in DESCRIBED_LAYERS}
top_10_described_sorted = sorted(described_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM DESCRIBED LAYERS ONLY (9, 17, 22, 29)")
print("=" * 70)
if top_10_described_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_described_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_described = [lf for lf, _ in top_10_described_sorted]

TOP 10 FROM DESCRIBED LAYERS ONLY (9, 17, 22, 29)
 1. L29 F 1784  influence=  7.1921
 2. L29 F 2081  influence=  7.1153
 3. L29 F  107  influence=  7.0198
 4. L29 F   80  influence=  5.9334
 5. L29 F 3347  influence=  5.9326
 6. L29 F 9459  influence=  5.7328
 7. L29 F11982  influence=  5.6974
 8. L29 F 6449  influence=  5.6277
 9. L29 F   33  influence=  5.5697
10. L29 F  440  influence=  5.3364


### get clerp descriptions from neuronpedia

In [22]:
import requests
import time

def fetch_clerps_batch(features_list, model_id="gemma-3-4b", slug_suffix="gemmascope-2-res-16k", delay=0.3):
    """Fetch clerp descriptions for a list of (layer, feature_idx) tuples."""
    clerps = {}
    
    for i, (layer, feat) in enumerate(features_list):
        source = f"{layer}-{slug_suffix}"
        url = f"https://www.neuronpedia.org/api/feature/{model_id}/{source}/{feat}"
        
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                exps = r.json().get("explanations", [])
                desc = exps[0].get("description") if exps else None
                clerps[(layer, feat)] = desc
                status = "✓" if desc else "✗ (no desc)"
            else:
                clerps[(layer, feat)] = None
                status = f"✗ (HTTP {r.status_code})"
        except Exception as e:
            clerps[(layer, feat)] = None
            status = f"✗ ({str(e)[:30]})"
        
        print(f"  [{i+1}/{len(features_list)}] L{layer:2d} F{feat:5d}: {status}")
        time.sleep(delay)
    
    return clerps

# Extract the top 10 described features
features_to_fetch = [lf for lf, _ in top_10_described_sorted]

# Fetch clerps
print("Fetching clerps for top 10 described features...")
clerps = fetch_clerps_batch(features_to_fetch)

# Print results
print("\n" + "="*70)
print("TOP 10 DESCRIBED FEATURES WITH CLERPS")
print("="*70)
for (layer, feat), score in top_10_described_sorted:
    desc = clerps.get((layer, feat))
    print(f"L{layer:2d} F{feat:5d} (influence={score:8.4f}): {desc or '[no description found]'}")

Fetching clerps for top 10 described features...
  [1/10] L29 F 1784: ✓
  [2/10] L29 F 2081: ✓
  [3/10] L29 F  107: ✓
  [4/10] L29 F   80: ✓
  [5/10] L29 F 3347: ✓
  [6/10] L29 F 9459: ✓
  [7/10] L29 F11982: ✓
  [8/10] L29 F 6449: ✓
  [9/10] L29 F   33: ✓
  [10/10] L29 F  440: ✓

TOP 10 DESCRIBED FEATURES WITH CLERPS
L29 F 1784 (influence=  7.1921): legal and financial penalties
L29 F 2081 (influence=  7.1153): marine organisms
L29 F  107 (influence=  7.0198): a adjective
L29 F   80 (influence=  5.9334): loneliness and negative emotions
L29 F 3347 (influence=  5.9326): variable assignments and numbers
L29 F 9459 (influence=  5.7328): rectangles
L29 F11982 (influence=  5.6974): `try` blocks for error handling
L29 F 6449 (influence=  5.6277): model refusal messages
L29 F   33 (influence=  5.5697): Italian subjects or cuisine
L29 F  440 (influence=  5.3364): Hebrew and Arabic words


### feature suppression one at a time (top five)

In [23]:
print("Suppressing one feature at a time to isolate effects...")

all_features = top_10_described
ablation_results = []

for layer, feat in all_features[:5]:  # just first 5 for speed
    single_intervention = [(layer, TARGET_POS, feat, 0.0)]
    
    try:
        ablated_text = model.feature_intervention_generate(
            prompt,
            single_intervention,
            do_sample=False,
            verbose=False
        )[0]
        
        # Check if output changed
        changed = ablated_text != pre_intervention_text
        status = "CHANGED" if changed else "unchanged"
        
        # Get clerp if available
        clerp_desc = clerps.get((layer, feat)) if (layer, feat) in clerps else None
        clerp_str = f" | {clerp_desc}" if clerp_desc else ""
        
        ablation_results.append({
            'layer': layer,
            'feat': feat,
            'changed': changed,
            'text': ablated_text,
            'clerp': clerp_desc
        })
        
        print(f"\nL{layer:2d} F{feat:5d}: {status}{clerp_str}")
        if changed:
            print(f"  → {ablated_text}")
    
    except Exception as e:
        print(f"L{layer:2d} F{feat:5d}: ERROR ({str(e)[:50]})")

Suppressing one feature at a time to isolate effects...

L29 F 1784: CHANGED | legal and financial penalties
  → A rhyming couplet:
He saw a carrot and had to grab it,
如果你想吃它,你必须把它拔出来

L29 F 2081: CHANGED | marine organisms
  → A rhyming couplet:
He saw a carrot and had to grab it,
sang the song of the rabbit.
He saw

L29 F  107: CHANGED | a adjective
  → A rhyming couplet:
He saw a carrot and had to grab it,
tle, tle, tle, tle, tle,

L29 F   80: CHANGED | loneliness and negative emotions
  → A rhyming couplet:
He saw a carrot and had to grab it,
 Advertiser
He saw a carrot and had to grab

L29 F 3347: CHANGED | variable assignments and numbers
  → A rhyming couplet:
He saw a carrot and had to grab it,
 Той побачив моркву і довелося


# try 30

In [ ]:
top_30_overall = sorted(all_features.items(), key=lambda x: -x[1])[:30]

In [27]:
print("=" * 70)
print("TOP 10 OVERALL (all layers)")
print("=" * 70)
for i, ((layer, feat), score) in enumerate(top_30_overall, 1):
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}  {in_desc}")

TOP 10 OVERALL (all layers)
 1. L29 F 1784  influence=  7.1921  ✓ DESCRIBED
 2. L32 F  156  influence=  7.1858    undescribed
 3. L29 F 2081  influence=  7.1153  ✓ DESCRIBED
 4. L29 F  107  influence=  7.0198  ✓ DESCRIBED
 5. L31 F 1432  influence=  6.9249    undescribed
 6. L28 F 3512  influence=  6.8427    undescribed
 7. L26 F 3121  influence=  6.7691    undescribed
 8. L31 F 5799  influence=  6.7016    undescribed
 9. L33 F  294  influence=  6.6289    undescribed
10. L33 F10133  influence=  6.5203    undescribed
11. L33 F 1372  influence=  6.4285    undescribed
12. L33 F  316  influence=  6.3146    undescribed
13. L31 F 1174  influence=  6.3012    undescribed
14. L28 F 5724  influence=  6.2846    undescribed
15. L31 F 5048  influence=  6.2528    undescribed
16. L27 F   90  influence=  6.2330    undescribed
17. L33 F 2378  influence=  6.1807    undescribed
18. L28 F 9222  influence=  6.0716    undescribed
19. L24 F 2294  influence=  6.0637    undescribed
20. L33 F11962  influence=  

In [28]:
features_to_intervene = [lf for lf, _ in top_30_overall]
intervention_tuples = [(layer, TARGET_POS, feat, 0.0) for layer, feat in features_to_intervene]

print("\n" + "=" * 70)
print("INTERVENTION TUPLES (top 30 features, suppressed at target position)")
print("=" * 70)
for i, (layer, pos, feat, val) in enumerate(intervention_tuples, 1):
    print(f"{i:2d}. Layer {layer:2d}, Position {pos:3d}, Feature {feat:5d}, Value {val}")


INTERVENTION TUPLES (top 30 features, suppressed at target position)
 1. Layer 29, Position  -1, Feature  1784, Value 0.0
 2. Layer 32, Position  -1, Feature   156, Value 0.0
 3. Layer 29, Position  -1, Feature  2081, Value 0.0
 4. Layer 29, Position  -1, Feature   107, Value 0.0
 5. Layer 31, Position  -1, Feature  1432, Value 0.0
 6. Layer 28, Position  -1, Feature  3512, Value 0.0
 7. Layer 26, Position  -1, Feature  3121, Value 0.0
 8. Layer 31, Position  -1, Feature  5799, Value 0.0
 9. Layer 33, Position  -1, Feature   294, Value 0.0
10. Layer 33, Position  -1, Feature 10133, Value 0.0
11. Layer 33, Position  -1, Feature  1372, Value 0.0
12. Layer 33, Position  -1, Feature   316, Value 0.0
13. Layer 31, Position  -1, Feature  1174, Value 0.0
14. Layer 28, Position  -1, Feature  5724, Value 0.0
15. Layer 31, Position  -1, Feature  5048, Value 0.0
16. Layer 27, Position  -1, Feature    90, Value 0.0
17. Layer 33, Position  -1, Feature  2378, Value 0.0
18. Layer 28, Position  -1, F

In [29]:
print("\n" + "=" * 70)
print("GENERATING WITH ORIGINAL FEATURES (no intervention)")
print("=" * 70)

pre_intervention_text = model.feature_intervention_generate(
    prompt,
    [],  # empty list = no interventions
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{pre_intervention_text}")

# ─────────────────────────────────────────────────────────────
# Generate with all 30 features suppressed
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("GENERATING WITH ALL 10 FEATURES SUPPRESSED")
print("=" * 70)

post_intervention_text = model.feature_intervention_generate(
    prompt,
    intervention_tuples,
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{post_intervention_text}")


GENERATING WITH ORIGINAL FEATURES (no intervention)


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
He ate it and then he had to crap it

GENERATING WITH ALL 10 FEATURES SUPPRESSED


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
vação,
vação,
vação,
vação


In [31]:
print("=" * 70)
print("PSEUDO-CLERP FOR TOP 30 OVERALL FEATURES")
print("=" * 70)

for i, ((layer, feat), score) in enumerate(top_30_overall, 1):
    tokens = pseudo_clerp_topk(model, layer, feat, tokenizer, top_k=10)
    
    # Check if this is a described feature
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    
    print(f"\n{i:2d}. L{layer:2d} F{feat:5d} (influence={score:8.4f}) {in_desc}")
    print(f"    Top tokens: {tokens}")

PSEUDO-CLERP FOR TOP 30 OVERALL FEATURES

 1. L29 F 1784 (influence=  7.1921) ✓ DESCRIBED
    Top tokens: [' Efter', ' För', ' Geb', ' Rie', ' Vä', ' Kas', 'Efter', ' Arbe', 'Kam', ' Stol']

 2. L32 F  156 (influence=  7.1858)   undescribed
    Top tokens: ['<unused338>', 'ꗜ', 'ri', '𒅤', ' PLDNN', '𒉶', 'rik', ' rid', '𒂷', 'MalayMarks']

 3. L29 F 2081 (influence=  7.1153) ✓ DESCRIBED
    Top tokens: ['Gre', '<strong>', '<sub>', ' Gre', ' gre', 'Gr', ' calon', ' Ref', ' ল', '<i>']

 4. L29 F  107 (influence=  7.0198) ✓ DESCRIBED
    Top tokens: ['<unused366>', '<unused687>', '<unused192>', '<unused201>', '<unused1166>', '<unused108>', '<unused1207>', 'Referências', '.</', '<unused712>']

 5. L31 F 1432 (influence=  6.9249)   undescribed
    Top tokens: [' פ', ' การ', ' ק', ' ס', ' ו', ' מ', ' ג', ' ש', ' כ', ' അ']

 6. L28 F 3512 (influence=  6.8427)   undescribed
    Top tokens: [' str', ' Lo', ' oznac', '條', 'が登場', ' Str', ' Card', 'MathML', ' Ministries', ' aggravate']

 7. L26 F 312

In [ ]:
pseudo_clerps = {
    (29, 1784): "Swedish and Northern European capitalized sentence-starting words.",
    (32, 156): "Cuneiform characters and ancient Sumerian/Akkadian script tokens.",
    (29, 2081): "HTML formatting tags and structural markup elements.",
    (29, 107): "Unused internal tokens and bibliography reference markers.",
    (31, 1432): "Hebrew alphabet characters and Thai script prefixes.",
    (28, 3512): "Mixed technical strings including string variables and MathML.",
    (26, 3121): "Mathematical slash symbols and fraction division delimiters.",
    (31, 5799): "Concrete singular nouns representing physical objects.",
    (33, 294): "Ancient Mesopotamian pictographic and cuneiform logograms.",
    (33, 10133): "Common multilingual word fragments and grammatical prefixes.",
    (33, 1372): "Single uppercase Latin letters used for lists or variables.",
    (33, 316): "Performance-related verbs and miscellaneous pharmaceutical or unused tokens.",
    (31, 1174): "Feminine pronouns and possessive adjectives across multiple languages.",
    (28, 5724): "Global terms for war and technical UI layout components.",
    (31, 5048): "Technical computing operations and mathematical plural nouns.",
    (27, 90): "Cuneiform logograms and reserved internal model tokens.",
    (33, 2378): "Adjectives describing negative social traits or personal mistakes.",
    (28, 9222): "Nouns related to religious buildings, weapons, and diverse objects.",
    (24, 2294): "Complex scientific, academic, and transition words in various languages.",
    (33, 11962): "Adverbs and adjectives signifying quality, adequacy, or completeness.",
    (33, 8717): "Bullet points, geometric symbols, and special character ornaments.",
    (30, 3908): "Financial market terminology focused on company stock and shares.",
    (28, 820): "Verbs of physical state and repeated punctuation marks.",
    (33, 10542): "Specialized scientific adjectives related to geology, biology, and physics.",
    (29, 80): "Multilingual variations of the word 'text' and size descriptors.",
    (29, 3347): "Suffixes and fragments of complex medical or academic terms.",
    (30, 14060): "Mathematical italicized 'f' variants and multilingual word starters.",
    (32, 9991): "Indefinite articles 'A' and 'An' across different fonts/scripts.",
    (32, 146): "Replacement characters and miscellaneous multilingual short fragments.",
    (33, 8973): "Abstract nouns representing administrative critiques, failings, or skepticism.",
}

In [32]:
print("Suppressing one feature at a time to isolate effects...")

ablation_results = []

for layer, feat in features_to_intervene:  # all 10 features
    single_intervention = [(layer, TARGET_POS, feat, 0.0)]
    
    try:
        ablated_text = model.feature_intervention_generate(
            prompt,
            single_intervention,
            do_sample=False,
            verbose=False
        )[0]
        
        # Check if output changed
        changed = ablated_text != pre_intervention_text
        status = "CHANGED" if changed else "unchanged"
        
        # Get clerp (try pseudo_clerps first, fall back to clerps)
        clerp_desc = pseudo_clerps.get((layer, feat)) or clerps.get((layer, feat))
        clerp_str = f" | {clerp_desc}" if clerp_desc else ""
        
        ablation_results.append({
            'layer': layer,
            'feat': feat,
            'changed': changed,
            'text': ablated_text,
            'clerp': clerp_desc
        })
        
        print(f"\nL{layer:2d} F{feat:5d}: {status}{clerp_str}")
        if changed:
            print(f"  → {ablated_text}")
    
    except Exception as e:
        print(f"L{layer:2d} F{feat:5d}: ERROR ({str(e)[:50]})")

Suppressing one feature at a time to isolate effects...

L29 F 1784: CHANGED | Swedish and Northern European capitalized sentence-starting words.
  → A rhyming couplet:
He saw a carrot and had to grab it,
如果你想吃它,你必须把它拔出来

L32 F  156: CHANGED | Cuneiform characters and ancient Sumerian/Akkadian script tokens.
  → A rhyming couplet:
He saw a carrot and had to grab it,

He ate it and then he had to crap

L29 F 2081: CHANGED | HTML formatting tags and structural markup elements.
  → A rhyming couplet:
He saw a carrot and had to grab it,
sang the song of the rabbit.
He saw

L29 F  107: CHANGED | Unused internal tokens and bibliography reference markers.
  → A rhyming couplet:
He saw a carrot and had to grab it,
tle, tle, tle, tle, tle,

L31 F 1432: CHANGED | Hebrew alphabet characters and Thai script prefixes.
  → A rhyming couplet:
He saw a carrot and had to grab it,
 Dichotomy:
The carrot was a carrot,

L28 F 3512: CHANGED | Mixed technical strings including string variables and MathML.
 